# Sentiment Analysis of Playstore App Using LSTM

This approach combines deep learning (LSTM) to extract meaningful features to classify sentiments.

In [1]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
import gensim.downloader as api

In [2]:
# Load Dataset & membuang nilai null
df = pd.read_csv('/content/sample_data/data_berlabel.csv', usecols=['polarity', 'text_akhir']).dropna()

In [3]:
# Menyeimbangkan Jumlah Dataset (Negative, Neutral, Positive)
df_majority = df[df['polarity'] == 'negative']
df_minority = [df[df['polarity'] == label] for label in ['neutral', 'positive']]
df_balanced = pd.concat(
    [df_majority] + [resample(df_, replace=True, n_samples=len(df_majority), random_state=42) for df_ in df_minority]
).sample(frac=1, random_state=42).reset_index(drop=True)

In [4]:
# Encode Labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df_balanced['polarity']).astype(np.int32)

In [5]:
# Optimized Text Preprocessing
clean_re = re.compile(r'http\S+|www\S+|[^a-zA-Z\s]')

def clean_text(text):
    return clean_re.sub('', text).lower().strip()

df_balanced['text_akhir'] = df_balanced['text_akhir'].astype(str).apply(clean_text)

In [6]:
# 🚀 Tokenization & Padding Optimization
OOV_TOK = "<OOV>"
max_length = 100

# Dynamically adjust vocab size
tokenizer = Tokenizer(oov_token=OOV_TOK)
tokenizer.fit_on_texts(df_balanced['text_akhir'])
vocab_size = len(tokenizer.word_index) + 1  # Use actual word count

X = pad_sequences(
    tokenizer.texts_to_sequences(df_balanced['text_akhir']),
    maxlen=max_length,
    padding='post',
    truncating='post',
    dtype=np.int32
)

In [7]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# Feature Extraction Using Word2Vec (Glove)
# Load GloVe Embeddings
embedding_dim = 50
word_vectors = api.load("glove-twitter-50")
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < vocab_size and word in word_vectors:
        embedding_matrix[i] = word_vectors[word]

[==================================================] 100.0% 199.5/199.5MB downloaded


In [9]:
# Optimized LSTM Model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], input_length=max_length, trainable=True),
    Bidirectional(LSTM(64, return_sequences=True, dropout=0.2)),  # Reduce units for faster training
    Bidirectional(LSTM(32, dropout=0.2)),  # Fewer LSTM units, better speed
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
# Compilation & Training with 10 Epochs
model.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(learning_rate=5e-4), metrics=['accuracy'])
model.fit(X_train, y_train, validation_split=0.2, epochs=10, batch_size=16, verbose=2)  # Reduce epochs for faster convergence

Epoch 1/10
416/416 - 79s - 191ms/step - accuracy: 0.4552 - loss: 1.0419 - val_accuracy: 0.5313 - val_loss: 0.9660
Epoch 2/10
416/416 - 82s - 197ms/step - accuracy: 0.5575 - loss: 0.9268 - val_accuracy: 0.6053 - val_loss: 0.8583
Epoch 3/10
416/416 - 74s - 177ms/step - accuracy: 0.6420 - loss: 0.7977 - val_accuracy: 0.6697 - val_loss: 0.7411
Epoch 4/10
416/416 - 80s - 192ms/step - accuracy: 0.7123 - loss: 0.6692 - val_accuracy: 0.7341 - val_loss: 0.6068
Epoch 5/10
416/416 - 83s - 200ms/step - accuracy: 0.7671 - loss: 0.5732 - val_accuracy: 0.7714 - val_loss: 0.5271
Epoch 6/10
416/416 - 71s - 171ms/step - accuracy: 0.8131 - loss: 0.4645 - val_accuracy: 0.8213 - val_loss: 0.4617
Epoch 7/10
416/416 - 69s - 167ms/step - accuracy: 0.8509 - loss: 0.3793 - val_accuracy: 0.8484 - val_loss: 0.4088
Epoch 8/10
416/416 - 85s - 204ms/step - accuracy: 0.8780 - loss: 0.3297 - val_accuracy: 0.8345 - val_loss: 0.4306
Epoch 9/10
416/416 - 82s - 196ms/step - accuracy: 0.9039 - loss: 0.2685 - val_accuracy: 

In [11]:
# Evaluate Model
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", round(test_acc, 4))

# Save Model
model.save("lstm_sentiment_model.keras")

Test Accuracy: 0.8657


In [12]:
# Optimized Prediction Function
def predict_sentiment(text, model, tokenizer, max_length=100):
    """
    Predict sentiment using LSTM model.
    """
    sequence = pad_sequences(
        tokenizer.texts_to_sequences([clean_text(text)]),
        maxlen=max_length,
        padding='post',
        truncating='post'
    )

    sentiment_label = np.argmax(model.predict_on_batch(sequence))  # Faster batch prediction
    return label_encoder.inverse_transform([sentiment_label])[0]

In [13]:
# Predict
loaded_model = tf.keras.models.load_model("lstm_sentiment_model.keras", compile=True)  # Ensure model is compiled
example_texts = [
    "Aplikasi parahhh....",
    "Lemot banget, nggak rekomended.",
    "cepet dan tampilannya keren."
]

for text in example_texts:
    print(f"Text: {text} → Predicted Sentiment: {predict_sentiment(text, loaded_model, tokenizer)}")

Text: Aplikasi parahhh.... → Predicted Sentiment: negative
Text: Lemot banget, nggak rekomended. → Predicted Sentiment: negative
Text: cepet dan tampilannya keren. → Predicted Sentiment: neutral
